# Benchmark
Comparison of performance and accuracy between the methods in the `numerical_methods` library.

## Cell 1 – Imports

In [8]:
import numpy as np
import sympy as sp
from time import perf_counter
from scipy.integrate import quad
from scipy.optimize._numdiff import approx_derivative
from scipy.optimize import brentq, approx_fprime

# Imported library already installed with pip.
import numerical_methods as nm

## Cell 3 – Available methods

In [9]:
integration_methods = {
    "Rectangle Rule": nm.rectangle_integrate,
    "Trapezoid Rule": nm.trapezoidal_integrate,
    "First Simpson Rule": nm.simpson1_integrate,
    "Second Simpson Rule": nm.simpson2_integrate,
    "Gauss-Legendre Quadrature": nm.gauss_legendre_integrate,
    "Monte Carlo": nm.monte_carlo_integrate
}

differentiation_methods = {
    "Forward Difference": nm.fd.forward,
    "Backward Difference": nm.fd.backward,
    "Central Difference": nm.fd.central,
    "Central Difference nth": nm.fd.central_nth,
    "Richardson Method": nm.richardson_derivative
}

root_methods = {
    "Bisection Method": nm.bisection_calculate,
    "Newton-Raphson Method": nm.newton_raphson_calculate,
    "Ridders Method": nm.ridders_calculate
}

series_methods = {
    "Taylor Series": nm.taylor_approx,
    "Fourier Series": nm.fourier_approx
}

linear_solvers = {
    "Linear System Solve": nm.linearsystem_solve
}

## Cell 4 – Numerical Integration

In [10]:
# Integration Interval
a = -np.pi
b = np.pi
L = (b - a) / 2
C = (a + b) / 2

# Interval Subdivisions or probability
n = 120

# Domain
x = np.linspace(a, b, 1000)

# Step
delta_x = 0.001

# Function
f = lambda x: np.cos(x) ** 2 + np.sin(2 * x)

scipy_reference = quad(f, a, b)[0]

print(f"{'Method':30}{'Result':>15}{'Error':>15}{'Time (ms)':>15}")
print("-"*75)

for name, method in integration_methods.items():
    start = perf_counter()
    result = method(f, a, b, n)
    elapsed = (perf_counter() - start) * 1000
    error = abs(result - scipy_reference)
    print(f"{name:30}{result:15.6f}{error:15.2e}{elapsed:15.4f}")

start = perf_counter()
scipy_reference = quad(f, a, b)[0]
elapsed = (perf_counter() - start) * 1000
print(f"{'Scipy Method':30}{scipy_reference:15.6f}{0.0:15.2e}{elapsed:15.4f}")

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Rectangle Rule                       3.141593       4.44e-16         0.1099
Trapezoid Rule                       3.141593       4.44e-16         0.2662
First Simpson Rule                   3.141593       4.44e-16         0.1361
Second Simpson Rule                  3.141593       4.44e-16         0.1218
Gauss-Legendre Quadrature            3.141593       2.09e-14         3.5695
Monte Carlo                          4.107345       9.66e-01         0.1584
Scipy Method                         3.141593       0.00e+00         0.0916


## Cell 5 – Numerical Differentiation

In [11]:
evaluation_point = 2
x_sym = sp.symbols('x')
f_sym = sp.cos(x_sym)**2 + sp.sin(2*x_sym)
df_sym = sp.diff(f_sym, x_sym)
reference_derivative = float(df_sym.subs(x_sym, evaluation_point))

print(f"{'Method':30}{'Result':>15}{'Error':>15}{'Time (ms)':>15}")
print("-"*75)

for name, method in differentiation_methods.items():
    start = perf_counter()
    derivative = method(f, evaluation_point, delta_x)
    elapsed = (perf_counter() - start) * 1000
    error = abs(derivative - reference_derivative)
    print(f"{name:30}{derivative:15.6f}{error:15.2e}{elapsed:15.4f}")

# SciPy: approx_fprime(x, f, h) -> grad
start = perf_counter()
derivative_scipy = approx_fprime(np.array([evaluation_point]), lambda v: f(v[0]), delta_x)[0]
elapsed = (perf_counter() - start) * 1000
error = abs(derivative_scipy - reference_derivative)
print(f"{'SciPy (approx_fprime)':30}{derivative_scipy:15.6f}{error:15.2e}{elapsed:15.4f}")

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Forward Difference                  -0.548317       2.17e-03         0.0603
Backward Difference                 -0.552652       2.17e-03         0.0063
Central Difference                  -0.550484       3.67e-07         0.0032
Central Difference nth              -0.550485       9.17e-08         0.0095
Richardson Method                   -0.550485       5.60e-14         0.0061
SciPy (approx_fprime)               -0.548317       2.17e-03         0.3679


## Cell 6 – Root Finding

In [12]:
# f has a sign change between -0.5 and -0.4 (bracket for Bisection / Ridders)
root_interval = (-0.5, -0.4)
x0_root = -0.45
max_iter = 100
tol = 1e-6

reference_root = float(sp.nsolve(f_sym, x_sym, x0_root))

print(f"{'Method':30}{'Result':>15}{'Error':>15}{'Time (ms)':>15}")
print("-"*75)

# Bisection: calculate(f, a, b, error) -> root
start = perf_counter()
root = nm.bisection_calculate(f, *root_interval, tol)
elapsed = (perf_counter() - start) * 1000
error = abs(root - reference_root)
print(f"{'Bisection Method':30}{root:15.6f}{error:15.2e}{elapsed:15.4f}")

# Newton-Raphson: calculate(f, x0, n, tolerance) -> (root, iterations)
start = perf_counter()
root, iterations = nm.newton_raphson_calculate(f, x0_root, max_iter, tol)
elapsed = (perf_counter() - start) * 1000
error = abs(root - reference_root)
print(f"{'Newton-Raphson Method':30}{root:15.6f}{error:15.2e}{elapsed:15.4f}")

# Ridders: calculate(f, a, b, n, tolerance) -> (root, iterations)
start = perf_counter()
root, iterations = nm.ridders_calculate(f, *root_interval, max_iter, tol)
elapsed = (perf_counter() - start) * 1000
error = abs(root - reference_root)
print(f"{'Ridders Method':30}{root:15.6f}{error:15.2e}{elapsed:15.4f}")

# SciPy Brent's Method: brentq(f, a, b, xtol) -> root
start = perf_counter()
root_scipy = brentq(f, *root_interval, xtol=tol)
elapsed = (perf_counter() - start) * 1000
error = abs(root_scipy - reference_root)
print(f"{'Brent Method (SciPy)':30}{root_scipy:15.6f}{error:15.2e}{elapsed:15.4f}")

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Bisection Method                    -0.463648       2.33e-07         0.1507
Newton-Raphson Method               -0.463648       3.92e-09         0.0918
Ridders Method                      -0.463648       1.81e-09         0.0802
Brent Method (SciPy)                -0.463648       1.59e-09         0.1376


## Cell 7 – Series Approximation

In [13]:
expansion_point = 0
order = 6
evaluation_x = 1.0
true_value = f(evaluation_x)

print(f"{'Method':30}{'Result':>15}{'Error':>15}{'Time (ms)':>15}")
print("-"*75)

start = perf_counter()
taylor_result = nm.taylor_approx(f, evaluation_x, expansion_point, order)
elapsed = (perf_counter() - start) * 1000
error = abs(taylor_result - true_value)
print(f"{'Taylor Series':30}{taylor_result:15.6f}{error:15.2e}{elapsed:15.4f}")

start = perf_counter()
fourier_result = nm.fourier_approx(f, evaluation_x, b, order)
elapsed = (perf_counter() - start) * 1000
error = abs(fourier_result - true_value)
print(f"{'Fourier Series':30}{fourier_result:15.6f}{error:15.2e}{elapsed:15.4f}")

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Taylor Series                        1.222223       2.10e-02         0.2587
Fourier Series                       1.201224       2.22e-16         3.4214


## Cell 8 – Linear Algebra

In [14]:
A = np.array([
    [10., 2., 1., 3., 0.],
    [2., 12., 2., 1., 4.],
    [1., 2., 15., 3., 2.],
    [3., 1., 3., 14., 5.],
    [0., 4., 2., 5., 13.]
])

b_vec = np.array([15., 20., 30., 25., 18.])

reference_solution = np.linalg.solve(A, b_vec)
reference_det = np.linalg.det(A)

print(f"{'Method':30}{'Result (norm)':>15}{'Error':>15}{'Time (ms)':>15}")
print("-" * 75)

# Gauss + Pivoting: gauss(matrix) -> (U, factors); pivoting(matrix) -> (A_permuted, p)

A_tilde = np.column_stack((A, b_vec))

start = perf_counter()
A_pivoted, p = nm.pivoting_elimination(A_tilde)
elapsed_pivot = (perf_counter() - start) * 1000

start = perf_counter()
U_tilde, factors = nm.gauss_elimination(A_pivoted)
elapsed_gauss = (perf_counter() - start) * 1000

U = U_tilde[:, :-1]
b_new = U_tilde[:, -1]
n = len(U)
x_manual = np.zeros(n)
for i in range(n - 1, -1, -1):
    soma = sum(U[i][k] * x_manual[k] for k in range(i + 1, n))
    x_manual[i] = b_new[i] - soma

error = np.linalg.norm(x_manual - reference_solution)
print(f"{'Gauss with Pivoting':30}{np.linalg.norm(x_manual):15.6f}{error:15.2e}{elapsed_pivot + elapsed_gauss:15.4f}")

# LU Decomposition (Crout): LU(matrix) -> L, U
start = perf_counter()
L, U = nm.lu_decomposition(A)
elapsed = (perf_counter() - start) * 1000
reconstruction_error = np.linalg.norm(L @ U - A)
print(f"{'LU Decomposition':30}{'—':>15}{reconstruction_error:15.2e}{elapsed:15.4f}")

# Determinant: calculate(matrix, method="gauss" | "lu") -> float

start = perf_counter()
det_gauss = nm.determinant_calculate(A, method = "gauss")
elapsed_gauss = (perf_counter() - start) * 1000
print(f"{'Determinant (Gauss)':30}{det_gauss:15.6f}{abs(det_gauss - reference_det):15.2e}{elapsed_gauss:15.4f}")

start = perf_counter()
det_lu = nm.determinant_calculate(A, method = "lu")
elapsed_lu = (perf_counter() - start) * 1000
print(f"{'Determinant (LU)':30}{det_lu:15.6f}{abs(det_lu - reference_det):15.2e}{elapsed_lu:15.4f}")

start = perf_counter()
det_numpy = np.linalg.det(A)
elapsed_numpy = (perf_counter() - start) * 1000
print(f"{'Determinant (NumPy)':30}{det_numpy:15.6f}{abs(det_numpy - reference_det):15.2e}{elapsed_numpy:15.4f}")

# Linear System Solve: solve(A, b, method="gauss" | "lu") -> x
start = perf_counter()
solution_gauss = nm.linearsystem_solve(A, b_vec, method = "gauss")
elapsed_gauss = (perf_counter() - start) * 1000
error_gauss = np.linalg.norm(solution_gauss - reference_solution)
print(f"{'Linear System (Gauss)':30}{np.linalg.norm(solution_gauss):15.6f}{error_gauss:15.2e}{elapsed_gauss:15.4f}")

start = perf_counter()
solution_lu = nm.linearsystem_solve(A, b_vec, method = "lu")
elapsed_lu = (perf_counter() - start) * 1000
error_lu = np.linalg.norm(solution_lu - reference_solution)
print(f"{'Linear System (LU)':30}{np.linalg.norm(solution_lu):15.6f}{error_lu:15.2e}{elapsed_lu:15.4f}")

start = perf_counter()
solution_cholesky = nm.linearsystem_solve(A, b_vec, method = "cholesky")
elapsed_cholesky = (perf_counter() - start) * 1000
error_cholesky = np.linalg.norm(solution_cholesky - reference_solution)
print(f"{'Linear System (Cholesky)':30}{np.linalg.norm(solution_cholesky):15.6f}{error_cholesky:15.2e}{elapsed_cholesky:15.4f}")

start = perf_counter()
solution_numpy = np.linalg.solve(A, b_vec)
elapsed_numpy = (perf_counter() - start) * 1000
error_numpy = np.linalg.norm(solution_numpy - reference_solution)
print(f"{'Linear System (NumPy)':30}{np.linalg.norm(solution_numpy):15.6f}{error_numpy:15.2e}{elapsed_numpy:15.4f}")

from scipy.optimize._numdiff import approx_derivative

# Jacobian: calculate(F, x, h) -> Jacobian Matrix
print("-" * 75)

def F(v):
    x_, y_ = v
    return np.array([x_**2 + y_**2 - 4, x_ - y_])

jacobian_point = np.array([1.0, 1.0])
h = 1e-6

# Sympy Reference for Jacobian
x_sym, y_sym = sp.symbols('x y')
F_sym = sp.Matrix([x_sym**2 + y_sym**2 - 4, x_sym - y_sym])
J_sym = F_sym.jacobian([x_sym, y_sym])
reference_jacobian = np.array(
    J_sym.subs({x_sym: jacobian_point[0], y_sym: jacobian_point[1]})
).astype(float)

start = perf_counter()
J = nm.jacobian_calculate(F, jacobian_point, h)
elapsed = (perf_counter() - start) * 1000
error = np.linalg.norm(J - reference_jacobian)
print(f"{'Jacobian':30}{'—':>15}{error:15.2e}{elapsed:15.4f}")
print(J)

start = perf_counter()
J_scipy = approx_derivative(F, jacobian_point)
elapsed_scipy = (perf_counter() - start) * 1000
error_scipy = np.linalg.norm(J_scipy - reference_jacobian)
print(f"{'Jacobian (SciPy)':30}{'—':>15}{error_scipy:15.2e}{elapsed_scipy:15.4f}")
print(J_scipy)

Method                          Result (norm)          Error      Time (ms)
---------------------------------------------------------------------------
Gauss with Pivoting                  2.329031       4.58e-16         0.3177
LU Decomposition                            —       4.44e-16         0.1785
Determinant (Gauss)             209655.000000       4.07e-10         0.2100
Determinant (LU)                209655.000000       3.78e-10         0.1588
Determinant (NumPy)             209655.000000       0.00e+00         0.1795
Linear System (Gauss)                2.329031       4.58e-16         0.4025
Linear System (LU)                   2.329031       3.85e-16         0.2612
Linear System (Cholesky)             2.329031       2.29e-16         0.1902
Linear System (NumPy)                2.329031       0.00e+00         0.3731
---------------------------------------------------------------------------
Jacobian                                    —       1.41e-06         0.1097
[[ 2.000001 